# Task 12 — Audit the Resume Screener for Fairness

**Purpose:** test counterfactual twin resumes, measure protected-group score gaps, and report ranking quality.

**Validation Vault rule:** the notebook first calculates and saves the twin-pair baseline. Only in a later cell does it load `unlock_CX-12_resume_protected_attrs.csv`. The protected attributes are used exclusively for auditing, never as model inputs.

In [1]:
from pathlib import Path
import re, json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

ROOT = Path("/workspace")
if not (ROOT / "data").exists():
    raise FileNotFoundError(f"Expected Docker-mounted project root at {ROOT}; data/ was not found.")

DATA_DIR = ROOT / "data"
REPORT_DIR = ROOT / "phase-3" / "reports"
VAULT_DIR = ROOT / "04_Validation_Vault"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load only resume/job data and counterfactual pairs at this stage.
# Protected attributes remain in the Validation Vault and are loaded only
# in the dedicated post-baseline audit cell below.
resume_df = pd.read_csv(DATA_DIR / "resumes.csv").fillna("")
job_df = pd.read_csv(DATA_DIR / "job_descriptions.csv").fillna("")
pairs_df = pd.read_csv(DATA_DIR / "resume_bias_pairs.csv").fillna("")

required_resume_cols = {
    "resume_id", "resume_text", "current_company", "total_experience_years",
    "target_job_family", "skills"
}
required_job_cols = {
    "job_id", "required_skills", "nice_to_have_skills",
    "min_experience_years", "description_text"
}
required_pair_cols = {
    "pair_id", "resume_id_original", "resume_id_twin",
    "skills_identical", "experience_identical", "attribute_changed"
}
assert required_resume_cols.issubset(resume_df.columns), (
    f"Missing resume columns: {sorted(required_resume_cols - set(resume_df.columns))}"
)
assert required_job_cols.issubset(job_df.columns), (
    f"Missing job columns: {sorted(required_job_cols - set(job_df.columns))}"
)
assert required_pair_cols.issubset(pairs_df.columns), (
    f"Missing twin-pair columns: {sorted(required_pair_cols - set(pairs_df.columns))}"
)
assert resume_df.resume_id.is_unique, "resume_id must be unique."
assert job_df.job_id.is_unique, "job_id must be unique."
assert pairs_df.pair_id.is_unique, "pair_id must be unique."
assert set(pairs_df.resume_id_original).issubset(set(resume_df.resume_id))
assert set(pairs_df.resume_id_twin).issubset(set(resume_df.resume_id))

print("Resumes:", resume_df.shape, "Jobs:", job_df.shape, "Twin pairs:", pairs_df.shape)


Resumes: (185, 15) Jobs: (10, 11) Twin pairs: (25, 8)


## 1. Reproduce the Task 11 scorer (no protected attributes loaded)

In [2]:
def normalize_text(value):
    value = str(value).lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

company_names = sorted({str(x).strip() for x in resume_df.current_company if str(x).strip()}, key=len, reverse=True)

def clean_resume_text(text):
    lines = str(text).splitlines()
    kept, section = [], None
    for line_no, line in enumerate(lines):
        value = line.strip()
        if not value:
            continue
        if line_no < 2:
            continue
        if re.fullmatch(r"(SUMMARY|EXPERIENCE|EXPERIENCE NOTES|SKILLS|EDUCATION|CERTIFICATIONS)", value, re.I):
            section = value.upper()
            continue
        if section == "EDUCATION":
            continue
        if "@" in value:
            continue
        value = re.sub(r"\bTier\s*[123]\s*institution\b", "", value, flags=re.I)
        for company in company_names:
            value = re.sub(re.escape(company), "", value, flags=re.I)
        if section == "EXPERIENCE" and not value.startswith("-") and "(" in value and ")" in value:
            match = re.match(r"^(.+?)[,|]\s*.+?\s*(\([0-9.]+\s*yrs?\))$", value, re.I)
            if match:
                value = f"{match.group(1)} {match.group(2)}"
        value = re.sub(r"\s+", " ", value).strip()
        if value:
            kept.append(value)
    return " ".join(kept)

resume_df["clean_resume_text"] = resume_df.resume_text.map(clean_resume_text)
job_skill_terms = sorted({
    skill.strip()
    for col in ["required_skills", "nice_to_have_skills"]
    for value in job_df[col]
    for skill in str(value).split(";")
    if skill.strip()
}, key=len, reverse=True)

def extract_skills(text):
    normalized = " " + normalize_text(text) + " "
    return {
        normalize_text(skill) for skill in job_skill_terms
        if (" " + normalize_text(skill) + " ") in normalized
    }

resume_df["extracted_skills"] = resume_df.clean_resume_text.map(extract_skills)
corpus = resume_df.clean_resume_text.tolist() + job_df.description_text.astype(str).tolist()
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
matrix = vectorizer.fit_transform(corpus)
resume_matrix = matrix[:len(resume_df)]
job_matrix = matrix[len(resume_df):]
cosine_matrix = cosine_similarity(job_matrix, resume_matrix)

resume_lookup = resume_df.set_index("resume_id")
job_lookup = job_df.set_index("job_id")
job_index = {job_id: i for i, job_id in enumerate(job_df.job_id)}

def score_resume(resume_id, job_id):
    resume = resume_lookup.loc[resume_id]
    job = job_lookup.loc[job_id]
    required = {normalize_text(x) for x in str(job.required_skills).split(";") if x.strip()}
    nice = {normalize_text(x) for x in str(job.nice_to_have_skills).split(";") if x.strip()}
    found = resume.extracted_skills
    required_coverage = len(required & found) / len(required) if required else 0.0
    nice_coverage = len(nice & found) / len(nice) if nice else 0.0
    experience_fit = min(float(resume.total_experience_years) / max(float(job.min_experience_years), 1.0), 1.0)
    resume_position = resume_df.index[resume_df.resume_id == resume_id][0]
    tfidf_similarity = float(cosine_matrix[job_index[job_id], resume_position])
    score = (0.50 * required_coverage + 0.15 * nice_coverage
             + 0.20 * tfidf_similarity + 0.15 * experience_fit)
    return {
        "match_score": score,
        "required_skill_coverage": required_coverage,
        "nice_skill_coverage": nice_coverage,
        "tfidf_similarity": tfidf_similarity,
        "experience_fit": experience_fit
    }

## 2. BEFORE-FIX twin-pair test — save this evidence before opening the Vault

In [3]:
assert pairs_df.pair_id.is_unique
assert pairs_df.skills_identical.eq("Y").all()
assert pairs_df.experience_identical.eq("Y").all()
assert set(pairs_df.resume_id_original).issubset(set(resume_df.resume_id))
assert set(pairs_df.resume_id_twin).issubset(set(resume_df.resume_id))

before_rows = []
for _, pair in pairs_df.iterrows():
    original = resume_lookup.loc[pair.resume_id_original]
    twin = resume_lookup.loc[pair.resume_id_twin]
    assert original.target_job_family == twin.target_job_family
    job_id = original.target_job_family
    a = score_resume(pair.resume_id_original, job_id)
    b = score_resume(pair.resume_id_twin, job_id)
    before_rows.append({
        "pair_id": pair.pair_id, "job_id": job_id,
        "resume_id_original": pair.resume_id_original,
        "resume_id_twin": pair.resume_id_twin,
        "attribute_changed": pair.attribute_changed,
        "original_score": a["match_score"], "twin_score": b["match_score"],
        "signed_gap": a["match_score"] - b["match_score"],
        "absolute_gap": abs(a["match_score"] - b["match_score"])
    })

before = pd.DataFrame(before_rows)
before_path = REPORT_DIR / "task12_before_fairness_pair_scores.csv"
before.to_csv(before_path, index=False)
print("Saved BEFORE-Vault baseline:", before_path)
print("Pairs:", len(before))
print("Mean absolute gap:", before.absolute_gap.mean())
print("Maximum absolute gap:", before.absolute_gap.max())
print("Non-zero gaps:", int((before.absolute_gap > 1e-12).sum()))
display(before)

Saved BEFORE-Vault baseline: /workspace/phase-3/reports/task12_before_fairness_pair_scores.csv
Pairs: 25
Mean absolute gap: 0.0
Maximum absolute gap: 0.0
Non-zero gaps: 0


,pair_id,job_id,resume_id_original,resume_id_twin,attribute_changed,original_score,twin_score,signed_gap,absolute_gap
0,BP-01,J01,RES-0123,RES-0161,candidate_name (gender-coded) and college tier,0.776746,0.776746,0.0,0.0
1,BP-02,J08,RES-0114,RES-0162,candidate_name (gender-coded) only,0.707476,0.707476,0.0,0.0
2,BP-03,J03,RES-0005,RES-0163,candidate_name (gender-coded) and college tier,0.454082,0.454082,0.0,0.0
3,BP-04,J03,RES-0117,RES-0164,candidate_name (gender-coded) only,0.315969,0.315969,0.0,0.0
4,BP-05,J07,RES-0118,RES-0165,candidate_name (gender-coded) and college tier,0.284245,0.284245,0.0,0.0
5,BP-06,J05,RES-0023,RES-0166,candidate_name (gender-coded) only,0.790528,0.790528,0.0,0.0
6,BP-07,J08,RES-0076,RES-0167,candidate_name (gender-coded) and college tier,0.762699,0.762699,0.0,0.0
7,BP-08,J02,RES-0026,RES-0168,candidate_name (gender-coded) only,0.271354,0.271354,0.0,0.0
8,BP-09,J09,RES-0007,RES-0169,candidate_name (gender-coded) and college tier,0.780927,0.780927,0.0,0.0
9,BP-10,J06,RES-0110,RES-0170,candidate_name (gender-coded) only,0.234759,0.234759,0.0,0.0


## 3. Validation Vault — open only after the baseline has been saved

The following cell is intentionally separate. Do not use the protected attributes in `score_resume`, feature extraction, TF-IDF, or ranking. They are only for post-score group analysis.

In [4]:
vault_path = VAULT_DIR / "unlock_CX-12_resume_protected_attrs.csv"
protected = pd.read_csv(vault_path)
assert protected.resume_id.is_unique
assert set(protected.resume_id) == set(resume_df.resume_id)
print("Vault loaded after baseline. Columns:", protected.columns.tolist())
display(protected.head())

Vault loaded after baseline. Columns: ['resume_id', 'inferred_gender', 'college_tier']


,resume_id,inferred_gender,college_tier
0,RES-0001,Male,Tier 3
1,RES-0002,Female,Tier 3
2,RES-0003,Female,Tier 3
3,RES-0004,Male,Tier 3
4,RES-0005,Female,Tier 2


## 4. Protected-group score gaps (audit only)

In [5]:
# Score every resume against its designated target job family.
target_rows = []
for _, resume in resume_df.iterrows():
    result = score_resume(resume.resume_id, resume.target_job_family)
    target_rows.append({
        "resume_id": resume.resume_id,
        "job_id": resume.target_job_family,
        "match_score": result["match_score"]
    })

target_scores = pd.DataFrame(target_rows).merge(
    protected, on="resume_id", validate="one_to_one"
)

def group_gap(scores, groups):
    means = pd.Series(scores).groupby(pd.Series(groups).values).mean()
    return means.to_dict(), float(means.max() - means.min())

gender_means, gender_gap = group_gap(target_scores.match_score, target_scores.inferred_gender)
college_means, college_gap = group_gap(target_scores.match_score, target_scores.college_tier)

group_rows = []
for attribute in ["inferred_gender", "college_tier"]:
    for group_name, group in target_scores.groupby(attribute):
        group_rows.append({
            "protected_attribute": attribute,
            "group": group_name,
            "resume_count": len(group),
            "mean_score": group.match_score.mean()
        })

group_summary = pd.DataFrame(group_rows)
group_summary.to_csv(REPORT_DIR / "task12_protected_group_score_summary.csv", index=False)
print("Gender group means:", gender_means, "gap:", gender_gap)
print("College-tier group means:", college_means, "gap:", college_gap)
display(group_summary)

Gender group means: {'Female': 0.543840685552731, 'Male': 0.523502500843039} gap: 0.020338184709692064
College-tier group means: {'Tier 1': 0.5525075098562533, 'Tier 2': 0.5039654069253887, 'Tier 3': 0.5461319453818524} gap: 0.048542102930864583


,protected_attribute,group,resume_count,mean_score
0,inferred_gender,Female,74,0.543841
1,inferred_gender,Male,111,0.523503
2,college_tier,Tier 1,49,0.552508
3,college_tier,Tier 2,71,0.503965
4,college_tier,Tier 3,65,0.546132


## 5. Ranking-quality comparison

In [6]:
# Recompute the Task 11 ranking metrics from human labels.
labels = pd.read_csv(DATA_DIR / "resume_job_labels.csv")
all_rank_rows = []
for _, job in job_df.iterrows():
    for _, resume in resume_df.iterrows():
        score = score_resume(resume.resume_id, job.job_id)["match_score"]
        all_rank_rows.append({
            "job_id": job.job_id, "resume_id": resume.resume_id,
            "match_score": score
        })
rankings = pd.DataFrame(all_rank_rows)
labeled = rankings.merge(labels, on=["resume_id", "job_id"], how="inner", validate="one_to_one")

def ndcg_at_k(relevance, k=5):
    relevance = np.asarray(relevance, dtype=float)[:k]
    discounts = np.log2(np.arange(2, len(relevance) + 2))
    dcg = float(np.sum((2 ** relevance - 1) / discounts))
    ideal = np.sort(np.asarray(relevance, dtype=float))[::-1][:k]
    ideal_discounts = np.log2(np.arange(2, len(ideal) + 2))
    idcg = float(np.sum((2 ** ideal - 1) / ideal_discounts))
    return dcg / idcg if idcg else 0.0

metric_rows = []
for job_id, group in labeled.groupby("job_id"):
    ordered = group.sort_values("match_score", ascending=False)
    top5 = ordered.head(5)
    total_shortlisted = int((ordered.shortlist == "Y").sum())
    metric_rows.append({
        "job_id": job_id,
        "precision_at_5": (top5.shortlist == "Y").mean(),
        "recall_at_5": (top5.shortlist == "Y").sum() / total_shortlisted if total_shortlisted else 0.0,
        "ndcg_at_5": ndcg_at_k(ordered.human_suitability_score.to_numpy(), 5),
        "spearman": spearmanr(ordered.match_score, ordered.human_suitability_score).statistic
    })
metrics = pd.DataFrame(metric_rows)
display(metrics)
print("Macro averages:")
display(metrics[["precision_at_5", "recall_at_5", "ndcg_at_5", "spearman"]].mean())

,job_id,precision_at_5,recall_at_5,ndcg_at_5,spearman
0,J01,1.0,0.714286,1.000000,0.830239
1,J02,0.4,0.666667,0.968906,0.860669
2,J03,1.0,0.416667,1.000000,0.832084
3,J04,1.0,0.312500,0.970210,0.933809
4,J05,1.0,0.555556,1.000000,0.803643
5,J06,1.0,0.833333,1.000000,0.614123
6,J07,1.0,0.625000,1.000000,0.821834
7,J08,1.0,0.555556,1.000000,0.831719
8,J09,1.0,0.357143,0.954171,0.905265
9,J10,1.0,0.454545,0.991771,0.831250


Macro averages:


precision_at_5    0.940000
recall_at_5       0.549125
ndcg_at_5         0.988506
spearman          0.826464
dtype: float64

## 6. Interpretation and fairness statement

Do not force group averages to match automatically. Group gaps can reflect different job-family composition and job-relevant qualifications. The controlled twin-pair result is the direct counterfactual test: if it is zero, do not change the scorer without another demonstrated source of unfairness.

State the exact tests, results, limitations, and that human review remains required.